# Stage 2 — One-Time Epoch Extraction (7GB raw -> ~400MB compact)

**What this does:** loops over every subject/run in your Drive folder, converts each
old-format MATLAB file via Octave, cuts out only the pre-response window around each
trial, downsamples, converts to float32, and appends to a single compact `.npz` file.
The 700MB raw files are never all held in memory at once.

**Run this ONCE.** Every later analysis loads the small `.npz`, not the raw data.

## Before you start: getting 7GB into Drive
- Google Drive free tier is 15GB, so 7GB fits.
- Easiest: install **Google Drive for Desktop**, drop the folder in, let it sync (one-time,
  runs in background). Or drag-and-drop the folder into drive.google.com.
- Keep the original folder structure; the code below searches recursively.
- Upload the `EEG_events_*.mat` files too (they're tiny and required).

In [ ]:
# 1. Setup: mount Drive, install Octave (needed because these are old MATLAB 5.0 files)
from google.colab import drive
drive.mount('/content/drive')

!apt-get update -qq
!apt-get install -y octave -qq
print("Octave installed")

In [ ]:
# 2. Configuration -- EDIT THESE
DATA_DIR = "/content/drive/MyDrive/PATH_TO_YOUR_EEG_FOLDER"   # <-- EDIT: folder containing the .mat files
OUT_PATH = "/content/drive/MyDrive/eeg_epochs_compact.npz"     # where the compact output goes

# Epoching parameters (chosen from the Stage 1 feasibility sweep)
# Chosen from the 24-subject feasibility analysis of the BIDS events.tsv timing data:
# 300/150 retains 99.5% of trials pooled and >=94.3% for EVERY subject (min sub-04).
# By contrast 400/150 drops sub-04 to 64% and 600/150 drops sub-22 to 24%, which would
# cripple those subjects as leave-one-subject-out test folds.
WINDOW_LEN_MS = 300      # window length in ms
BUFFER_MS     = 150      # window ends this many ms before each trial's own response
ORIG_FS       = 1000     # original sampling rate (Hz)
TARGET_FS     = 250      # downsample to this (Hz); 250 is ample for CPP/Pe-scale signals

DOWNSAMPLE_FACTOR = ORIG_FS // TARGET_FS
N_SAMPLES_OUT = int(WINDOW_LEN_MS / 1000 * TARGET_FS)

print(f"Window: {WINDOW_LEN_MS} ms ending {BUFFER_MS} ms pre-response")
print(f"Downsample {ORIG_FS} -> {TARGET_FS} Hz (factor {DOWNSAMPLE_FACTOR})")
print(f"Output samples per trial: {N_SAMPLES_OUT}")

In [ ]:
# 3. Find and pair up the data/events files
import os, glob, re

data_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "EEG_data_sub-*.mat"), recursive=True))
event_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "EEG_events_sub-*.mat"), recursive=True))

pattern = re.compile(r"sub-(\d+)_run-(\d+)")

def key_of(path):
    m = pattern.search(os.path.basename(path))
    return (m.group(1), m.group(2)) if m else None

data_map  = {key_of(f): f for f in data_files if key_of(f)}
event_map = {key_of(f): f for f in event_files if key_of(f)}

paired = sorted(set(data_map.keys()) & set(event_map.keys()))
missing_events = sorted(set(data_map.keys()) - set(event_map.keys()))
missing_data   = sorted(set(event_map.keys()) - set(data_map.keys()))

print(f"Found {len(data_files)} data files, {len(event_files)} events files")
print(f"Complete pairs to process: {len(paired)}")
for k in paired:
    print(f"  sub-{k[0]} run-{k[1]}")
if missing_events:
    print(f"\nWARNING - data file but no events file: {missing_events}")
if missing_data:
    print(f"WARNING - events file but no data file (skipped): {missing_data}")

In [ ]:
# 4. Helper: load one raw EEG file via Octave into a numpy array
import subprocess
import numpy as np
from scipy.io import loadmat

def load_eeg_via_octave(mat_path, tmp_out="/content/tmp_Y.mat"):
    """Old MATLAB 5.0 structs can't be read by scipy directly, so route through Octave."""
    if os.path.exists(tmp_out):
        os.remove(tmp_out)
    cmd = (
        f"load('{mat_path}', 'EEGdata'); Y=EEGdata.Y; "
        f"save('-mat7-binary', '{tmp_out}', 'Y');"
    )
    result = subprocess.run(
        ["octave", "--no-gui", "--quiet", "--eval", cmd],
        capture_output=True, text=True
    )
    if not os.path.exists(tmp_out):
        raise RuntimeError(f"Octave failed for {mat_path}:\n{result.stderr[:500]}")
    Y = loadmat(tmp_out)["Y"]
    os.remove(tmp_out)
    return Y

In [ ]:
# 5. MAIN LOOP: epoch every subject/run, accumulate compact arrays
# Processes one file at a time and frees it before the next, so memory stays flat.
import gc, time

X_all, subj_all, run_all = [], [], []
conf_all, acc_all, choice_all, dotdir_all, rt_all, tidx_all = [], [], [], [], [], []
skipped_log = []

t_start = time.time()

for i, key in enumerate(paired):
    subj, run = key
    print(f"[{i+1}/{len(paired)}] sub-{subj} run-{run} ...", end=" ", flush=True)

    try:
        ev = loadmat(event_map[key])
        tstim = ev["tstim"].squeeze().astype(float)
        tresp = ev["tresp"].squeeze().astype(float)
        # NOTE: use the RT field (authoritative) rather than tresp-tstim. In the BIDS
        # events.tsv the two disagree by up to ~84 ms due to rounded timestamps.
        confidence = ev["confidence"].squeeze().astype(int)
        accuracy   = ev["accuracy"].squeeze().astype(int)
        choice     = ev["choice"].squeeze().astype(int)
        dotdir     = ev["dotdirection"].squeeze().astype(int)
        RT         = ev["RT"].squeeze().astype(float)

        EEG = load_eeg_via_octave(data_map[key])
        n_ch, n_samp = EEG.shape

        n_trials = len(tstim)
        n_kept, n_skip = 0, 0

        for t in range(n_trials):
            # Define the window from tstim + RT (authoritative) rather than tresp,
            # so the window position is not affected by timestamp rounding.
            resp_time = tstim[t] + RT[t]
            win_end   = int(round(resp_time - BUFFER_MS))
            win_start = win_end - WINDOW_LEN_MS

            # validity: must sit inside the recording AND not start before stimulus onset
            if win_start < 0 or win_end > n_samp or win_start < int(tstim[t]):
                n_skip += 1
                continue

            seg = EEG[:, win_start:win_end]           # (n_ch, WINDOW_LEN_MS) at 1000 Hz
            if seg.shape[1] != WINDOW_LEN_MS:
                n_skip += 1
                continue

            # decimate by simple block-averaging (acts as an anti-alias low-pass)
            seg = seg[:, :DOWNSAMPLE_FACTOR * N_SAMPLES_OUT]
            seg = seg.reshape(n_ch, N_SAMPLES_OUT, DOWNSAMPLE_FACTOR).mean(axis=2)

            X_all.append(seg.astype(np.float32))
            subj_all.append(subj); run_all.append(run)
            conf_all.append(confidence[t]); acc_all.append(accuracy[t])
            choice_all.append(choice[t]); dotdir_all.append(dotdir[t])
            rt_all.append(RT[t]); tidx_all.append(t)
            n_kept += 1

        skipped_log.append((subj, run, n_trials, n_kept, n_skip))
        print(f"kept {n_kept}/{n_trials} (skipped {n_skip}), channels={n_ch}")

        del EEG
        gc.collect()

    except Exception as e:
        print(f"FAILED: {e}")
        skipped_log.append((subj, run, -1, 0, -1))

print(f"\nTotal elapsed: {(time.time()-t_start)/60:.1f} min")
print(f"Total epochs collected: {len(X_all)}")

In [ ]:
# 6. Check channel-count consistency, then stack and save
ch_counts = sorted(set(x.shape[0] for x in X_all))
print("Distinct channel counts across epochs:", ch_counts)

if len(ch_counts) > 1:
    print("\nNOTE: subjects have differing channel counts (the dataset excludes bad channels")
    print("per subject). Cropping all epochs to the minimum common count is a crude fix that")
    print("may misalign electrodes across subjects. Report this and consider per-subject")
    print("decoding, or channel-name-based alignment, before pooling across subjects.")
    min_ch = ch_counts[0]
    X_all = [x[:min_ch, :] for x in X_all]
    print(f"Cropped all epochs to {min_ch} channels.")

X = np.stack(X_all)                      # (n_epochs, n_channels, n_samples)
print("\nFinal X shape:", X.shape, X.dtype)
print(f"In-memory size: {X.nbytes/1e6:.1f} MB")

np.savez_compressed(
    OUT_PATH,
    X=X,
    subject=np.array(subj_all),
    run=np.array(run_all),
    confidence=np.array(conf_all),
    accuracy=np.array(acc_all),
    choice=np.array(choice_all),
    dotdirection=np.array(dotdir_all),
    RT=np.array(rt_all),
    trial_index=np.array(tidx_all),
    window_len_ms=WINDOW_LEN_MS,
    buffer_ms=BUFFER_MS,
    fs=TARGET_FS,
)

print(f"\nSaved to {OUT_PATH}")
print(f"File size on disk: {os.path.getsize(OUT_PATH)/1e6:.1f} MB")

In [ ]:
# 7. Summary report -- paste this output back into the chat
import numpy as np
from collections import Counter

print("=== PER-RUN TRIAL RETENTION ===")
print(f"{'sub':>5} {'run':>5} {'total':>7} {'kept':>6} {'skipped':>8}")
for subj, run, tot, kept, skip in skipped_log:
    print(f"{subj:>5} {run:>5} {tot:>7} {kept:>6} {skip:>8}")

subj_arr = np.array(subj_all)
acc_arr  = np.array(acc_all)
conf_arr = np.array(conf_all)
choice_arr = np.array(choice_all)

n_total = len(acc_arr)
n_subjects = len(set(subj_all))
print(f"\n=== POOLED TOTALS ===")
print(f"Epochs: {n_total}   Subjects: {n_subjects}   Runs: {len(set(zip(subj_all, run_all)))}")

print("\n=== CHOICE BALANCE ===")
for v, c in sorted(Counter(choice_arr).items()):
    print(f"  choice={v}: {c} ({100*c/n_total:.1f}%)")

print("\n=== ACCURACY BALANCE ===")
for v, c in sorted(Counter(acc_arr).items()):
    print(f"  accuracy={v}: {c} ({100*c/n_total:.1f}%)")

low_labels, high_labels = [1,2,3], [7,8,9]
is_low  = np.isin(conf_arr, low_labels)
is_high = np.isin(conf_arr, high_labels)
print(f"\n=== CONFIDENCE SPLIT ===")
print(f"  Low(1-3): {is_low.sum()}   Medium(4-6): {(~is_low & ~is_high).sum()}   High(7-9): {is_high.sum()}")

print(f"\n=== 4-WAY ACCURACY x CONFIDENCE (the key table) ===")
print(f"{'':>12} {'Low conf':>10} {'High conf':>10}")
for av, al in [(1,'Correct'), (0,'Incorrect')]:
    m = acc_arr == av
    print(f"{al:>12} {int((m&is_low).sum()):>10} {int((m&is_high).sum()):>10}")

oc = (acc_arr == 0) & is_high
print(f"\n>>> Overconfidence cell (incorrect + high conf): {int(oc.sum())} trials")
print(f">>> Contributed by {len(set(subj_arr[oc]))} distinct subjects")
print(">>> Per-subject counts in that cell:")
for s, c in sorted(Counter(subj_arr[oc]).items()):
    print(f"      sub-{s}: {c}")

## What to send back

Paste the output of **cell 7** (and cell 6's channel-count line). That gives us:
- pooled trial counts and per-run retention at the chosen window
- whether the overconfidence cell is now large enough AND spread across enough subjects
- whether channel counts differ across subjects (which affects whether we can pool at all)

Once we have that, the numbers go straight into Section 4.1 / Table 2 of the paper draft,
and we move to the actual decoding (RQ1/RQ2).